In [9]:
import os
from langchain_groq import ChatGroq
from langchain.schema.output_parser import StrOutputParser
from langchain.prompts import ChatPromptTemplate
from dotenv import load_dotenv

import rich

In [5]:
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGHAIN_TRACING_V2"] = "true"

In [21]:
model = ChatGroq(model = "llama-3.3-70b-versatile")

prompt = ChatPromptTemplate.from_template("Tell me a joke about {topic}")
# print(prompt)

output_parser = StrOutputParser()

chain = prompt | model | output_parser

In [22]:
chain.input_schema.model_json_schema()

{'properties': {'topic': {'title': 'Topic', 'type': 'string'}},
 'required': ['topic'],
 'title': 'PromptInput',
 'type': 'object'}

In [23]:
prompt.input_schema.model_json_schema()

{'properties': {'topic': {'title': 'Topic', 'type': 'string'}},
 'required': ['topic'],
 'title': 'PromptInput',
 'type': 'object'}

In [24]:
model.input_schema.model_json_schema()

{'$defs': {'AIMessage': {'additionalProperties': True,
   'description': 'Message from an AI.\n\nAIMessage is returned from a chat model as a response to a prompt.\n\nThis message represents the output of the model and consists of both\nthe raw output as returned by the model together standardized fields\n(e.g., tool calls, usage metadata) added by the LangChain framework.',
   'properties': {'content': {'anyOf': [{'type': 'string'},
      {'items': {'anyOf': [{'type': 'string'}, {'type': 'object'}]},
       'type': 'array'}],
     'title': 'Content'},
    'additional_kwargs': {'title': 'Additional Kwargs', 'type': 'object'},
    'response_metadata': {'title': 'Response Metadata', 'type': 'object'},
    'type': {'const': 'ai',
     'default': 'ai',
     'title': 'Type',
     'type': 'string'},
    'name': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
     'default': None,
     'title': 'Name'},
    'id': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
     'default': None,
     '

In [25]:
output_parser.input_schema.model_json_schema()

{'$defs': {'AIMessage': {'additionalProperties': True,
   'description': 'Message from an AI.\n\nAIMessage is returned from a chat model as a response to a prompt.\n\nThis message represents the output of the model and consists of both\nthe raw output as returned by the model together standardized fields\n(e.g., tool calls, usage metadata) added by the LangChain framework.',
   'properties': {'content': {'anyOf': [{'type': 'string'},
      {'items': {'anyOf': [{'type': 'string'}, {'type': 'object'}]},
       'type': 'array'}],
     'title': 'Content'},
    'additional_kwargs': {'title': 'Additional Kwargs', 'type': 'object'},
    'response_metadata': {'title': 'Response Metadata', 'type': 'object'},
    'type': {'const': 'ai',
     'default': 'ai',
     'title': 'Type',
     'type': 'string'},
    'name': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
     'default': None,
     'title': 'Name'},
    'id': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
     'default': None,
     '

In [26]:
output_parser.output_schema.model_json_schema()

{'title': 'StrOutputParserOutput', 'type': 'string'}

In [28]:
# for s in chain.stream({"topic":"bears"}):
#     print(s.content, end="", flush=True)

In [29]:
await chain.ainvoke({"topic": "bear"})

'Why did the bear go to the doctor?\n\nBecause it had a grizzly cough!'

## Prompt

In [30]:
model = ChatGroq(model = "llama-3.3-70b-versatile")

prompt = ChatPromptTemplate.from_template("Tell me a joke about {topic}")

chain = prompt | model

## Add a stop Sequence

In [34]:
chain = prompt | model.bind(stop=["\n"])  # Stop at the new lines

response = chain.invoke({"topic":"bears"})
print(response.content)

Why did the bear go to the doctor?


## LCEL with RAG

In [41]:
from operator import itemgetter
# from langchain.embeddings import OpenAIEmbeddings
# from langchain.schema.runnable import RunnableLembda, RunnablePassthough
from langchain_core.runnables.base import RunnableLambda
from langchain_core.runnables.passthrough import RunnablePassthrough
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

### Setup vectorstore

In [44]:
# Load embeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Create FAISS vectorstore
vectorstore = FAISS.from_texts(
    ["merk likes EDM music", "merk likes working out"], 
    embedding=embedding_model
)

retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

chain = (
    {"context":retriever, "question":RunnablePassthrough()} | prompt
    | model | StrOutputParser()
)

In [45]:
chain.invoke("What kind of music does merk like?")

'Merk likes EDM music.'

https://www.youtube.com/watch?v=moJRxxEddzU